<a href="https://colab.research.google.com/github/mostofa89/Pytorch_Fundamentals/blob/main/Multi_Variable_Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
torch.manual_seed(42)
n = 25

x1 = torch.randn(n) * 2 + 5
x2 = torch.randn(n) * 3 + 8
x3 = torch.rand(n) * 10
x4 = torch.randn(n) * 4

y = 3*x1 + 2*x2 - 1.5*x3 + 0.5*x4 + 4 + torch.randn(n)

In [ ]:
def plot_datapoints(x1 = x1, x2 = x2, x3 = x3, x4 = x4, y = y):
  plt.scatter(x1, y, label="x1")
  plt.scatter(x2, y, label="x2")
  plt.scatter(x3, y, label="x3")
  plt.scatter(x4, y, label="x4")

  plt.xlabel("X")
  plt.ylabel("Y")
  plt.title("X vs Y")
  plt.legend()
  plt.grid(True)

  plt.show()

In [ ]:
plot_datapoints()

In [ ]:
class LinearRegression(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear_layer = nn.Linear(in_features = 4, out_features = 1)


  def forward(self, x : torch.Tensor) -> torch.Tensor:
    return self.linear_layer(x)


torch.manual_seed(42)
model_1 = LinearRegression()
model_1.state_dict()

In [ ]:
X = torch.stack([x1, x2, x3, x4], dim=1).float()
y = y.float()

predictions = model_1(X)

print(predictions)

In [ ]:
train_split = int(len(X) * 0.8)

X_train, X_test, y_train, y_test = X[ : train_split], X[train_split : ],  y[ : train_split], y[train_split : ]

len(X_train), len(X_test), len(y_train), len(y_test)

In [ ]:
weights = model_1.linear_layer.weight.detach()
biases = model_1.linear_layer.bias.detach()

w1, w2, w3, w4 = weights[0]
bias = biases[0]

print("w1 =", w1)
print("w2 =", w2)
print("w3 =", w3)
print("w4 =", w4)
print("bias =", bias)

In [ ]:
x1_range = torch.linspace(x1.min(), x1.max(), 20)
x2_range = torch.linspace(x2.min(), x2.max(), 20)

X1_grid, X2_grid = torch.meshgrid(
    x1_range,
    x2_range,
    indexing="ij"
)

x3_fixed = x3.float().mean()
x4_fixed = x4.float().mean()

Y_plane = (
    w1 * X1_grid
    + w2 * X2_grid
    + w3 * x3_fixed
    + w4 * x4_fixed
    + bias
)

In [ ]:
fig = plt.figure(figsize=(10, 7))

ax = fig.add_subplot(111, projection="3d")

# Regression plane
ax.plot_surface(
    X1_grid.numpy(),
    X2_grid.numpy(),
    Y_plane.numpy(),
    alpha=0.5
)

# Actual data points
ax.scatter(
    x1.numpy(),
    x2.numpy(),
    y.numpy(),
    s=80,
    label="Actual data"
)

ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_zlabel("y")

ax.set_title("3D Linear Regression Plane")

ax.legend()

plt.show()

In [ ]:
loss_func = nn.L1Loss()
optimizer = torch.optim.Adam(params = model_1.parameters(), lr = 0.05)

In [ ]:
epoch_count = []
loss_values = []
test_loss_values = []

# Store minimum losses
min_train_loss = float('inf')
min_test_loss = float('inf')

# Store epochs where minimum loss occurs
min_train_epoch = 0
min_test_epoch = 0

epochs = 500

for epoch in range(epochs):

    # -------------------
    # Training
    # -------------------
    model_1.train()

    y_pred = model_1(X_train)

    loss = loss_func(y_pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # -------------------
    # Testing
    # -------------------
    model_1.eval()

    with torch.inference_mode():
        test_pred = model_1(X_test)
        test_loss = loss_func(test_pred, y_test)

    # Convert loss to Python number
    train_loss = loss.item()
    test_loss_value = test_loss.item()

    # -------------------
    # Store all values
    # -------------------
    epoch_count.append(epoch)
    loss_values.append(train_loss)
    test_loss_values.append(test_loss_value)

    # -------------------
    # Store minimum loss
    # -------------------
    if train_loss < min_train_loss:
        min_train_loss = train_loss
        min_train_epoch = epoch

    if test_loss_value < min_test_loss:
        min_test_loss = test_loss_value
        min_test_epoch = epoch

    # -------------------
    # Print every 5 epochs
    # -------------------
    if epoch % 20 == 0:
        print(
            f"Epoch: {epoch} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Test Loss: {test_loss_value:.4f}"
        )


# -------------------
# Final minimum losses
# -------------------

print("\nMinimum Loss Results")
print("--------------------")
print(f"Minimum Train Loss: {min_train_loss:.4f}")
print(f"At Epoch: {min_train_epoch}")

print(f"Minimum Test Loss: {min_test_loss:.4f}")
print(f"At Epoch: {min_test_epoch}")

In [ ]:
plt.plot(epoch_count, loss_values, label='Train Loss')
plt.plot(epoch_count, test_loss_values, label='Test Loss')

plt.title('Train and Test Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
X = torch.stack([x1, x2, x3, x4], dim=1).float()
y = y.float()
y_col = y.unsqueeze(1)          # shape (4, 1), used for training/loss

train_split = int(len(X) * 0.8)
X_train, X_test = X[:train_split], X[train_split:]
y_train, y_test = y_col[:train_split], y_col[train_split:]

In [ ]:
model_1.eval()
with torch.inference_mode():
    train_pred = model_1(X_train).squeeze()
    test_pred  = model_1(X_test).squeeze()
    all_pred   = model_1(X).squeeze()

# ---- Plot 1: predicted vs actual ----
plt.figure(figsize=(7, 6))
plt.scatter(y_train.squeeze().numpy(), train_pred.numpy(), s=90, label="Train")
plt.scatter(y_test.squeeze().numpy(),  test_pred.numpy(),  s=90, marker="s", label="Test")

lo = min(y.min().item(), all_pred.min().item()) - 5
hi = max(y.max().item(), all_pred.max().item()) + 5
plt.plot([lo, hi], [lo, hi], "k--", label="Perfect prediction")

plt.xlabel("Actual y")
plt.ylabel("Predicted y")
plt.title("Predicted vs Actual (after training)")
plt.legend()
plt.grid(True)
plt.show()

# ---- Plot 2: per-sample comparison ----
idx = np.arange(len(y))
plt.figure(figsize=(20, 12))
plt.bar(idx - 0.2, y.numpy(), width=0.4, label="Actual")
plt.bar(idx + 0.2, all_pred.numpy(), width=0.4, label="Predicted")
plt.xticks(idx, [f"sample {i}" for i in idx])
plt.ylabel("y")
plt.title("Actual vs Predicted per sample")
plt.legend()
plt.show()

In [ ]:
w = model_1.linear_layer.weight.detach()[0]
b = model_1.linear_layer.bias.detach()[0]

Y_plane = (w[0] * X1_grid + w[1] * X2_grid
           + w[2] * x3.float().mean() + w[3] * x4.float().mean() + b)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(X1_grid.numpy(), X2_grid.numpy(), Y_plane.numpy(), alpha=0.5)
ax.scatter(x1.numpy(), x2.numpy(), y.numpy(), s=80, color="red", label="Actual")
ax.scatter(x1.numpy(), x2.numpy(), all_pred.numpy(), s=80, color="green", marker="^", label="Predicted")
ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.set_zlabel("y")
ax.set_title("Trained Regression Plane (x3, x4 held at their means)")
ax.legend()
plt.show()